## 0. Setup & Configuration

In [ ]:
import os

import pandas as pd
import numpy as np
import yfinance as yf
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ============================================================
# MODEL REGISTRY
# ============================================================
# We compare ChronoBERT checkpoints at different temporal cutoffs
# against standard baselines (BERT, FinBERT).
# Key idea: ChronoBERT_2017/2018 are "chronologically consistent"
# for 2019-2022 news, while ChronoBERT_2021/2022 have look-ahead.

MODELS = {
    "ChronoBERT_2017": "manelalab/chrono-bert-v1-20171231",
    "ChronoBERT_2018": "manelalab/chrono-bert-v1-20181231",
    "ChronoBERT_2021": "manelalab/chrono-bert-v1-20211231",
    "ChronoBERT_2022": "manelalab/chrono-bert-v1-20221231",
    "Standard_BERT":   "bert-base-uncased",
    "FinBERT":         "yiyanghkust/finbert-pretrain"
}

# Classify models by temporal consistency w.r.t. the 2019-2022 test period
CHRONO_CONSISTENT = ["ChronoBERT_2017", "ChronoBERT_2018"]  # trained BEFORE test period
CHRONO_INCONSISTENT = ["ChronoBERT_2021", "ChronoBERT_2022"]  # trained DURING/AFTER
BASELINES = ["Standard_BERT", "FinBERT"]

# Colour palette for consistent plotting
MODEL_COLORS = {
    "ChronoBERT_2017": "#1b9e77",
    "ChronoBERT_2018": "#66c2a5",
    "ChronoBERT_2021": "#d95f02",
    "ChronoBERT_2022": "#fc8d62",
    "Standard_BERT":   "#7570b3",
    "FinBERT":         "#b2abd2",
}

# Sector mapping for the COVID-affected stock universe
SECTOR_MAP = {
    # Tech / Stay-at-home
    'ZM': 'Tech', 'PTON': 'Tech', 'DOCU': 'Tech', 'NFLX': 'Tech', 'AMZN': 'Tech',
    'SHOP': 'Tech', 'TDOC': 'Tech', 'NVDA': 'Tech', 'AMD': 'Tech', 'AAPL': 'Tech',
    'MSFT': 'Tech', 'GOOGL': 'Tech', 'META': 'Tech', 'ETSY': 'Tech', 'CHWY': 'Tech',
    'W': 'Tech', 'ROKU': 'Tech', 'PYPL': 'Tech', 'SQ': 'Tech', 'TEAM': 'Tech',
    'CRWD': 'Tech', 'OKTA': 'Tech', 'ZS': 'Tech', 'TWLO': 'Tech', 'DDOG': 'Tech',
    'FSLY': 'Tech', 'NET': 'Tech', 'SNOW': 'Tech', 'EBAY': 'Tech', 'CRM': 'Tech',
    # Consumer Staples / Retail
    'TGT': 'Retail', 'WMT': 'Retail', 'COST': 'Retail', 'HD': 'Retail', 'LOW': 'Retail',
    'CLX': 'Retail', 'PG': 'Retail', 'KMB': 'Retail', 'DPZ': 'Retail',
    'GPS': 'Retail', 'M': 'Retail', 'JWN': 'Retail', 'KSS': 'Retail', 'URBN': 'Retail',
    # Healthcare / Pharma
    'MRNA': 'Healthcare', 'PFE': 'Healthcare', 'BNTX': 'Healthcare', 'JNJ': 'Healthcare',
    'GILD': 'Healthcare', 'REGN': 'Healthcare', 'ABT': 'Healthcare', 'LLY': 'Healthcare',
    'AZN': 'Healthcare', 'NVAX': 'Healthcare',
    # Travel / Leisure / Hospitality
    'CCL': 'Travel', 'RCL': 'Travel', 'NCLH': 'Travel', 'DAL': 'Travel', 'UAL': 'Travel',
    'AAL': 'Travel', 'LUV': 'Travel', 'BA': 'Travel', 'MAR': 'Travel', 'HLT': 'Travel',
    'EXPE': 'Travel', 'BKNG': 'Travel', 'ABNB': 'Travel', 'UBER': 'Travel', 'LYFT': 'Travel',
    'DIS': 'Travel', 'AMC': 'Travel', 'CNK': 'Travel', 'LYV': 'Travel', 'MGM': 'Travel',
    'CZR': 'Travel', 'WYNN': 'Travel', 'LVS': 'Travel', 'DRI': 'Travel', 'PLAY': 'Travel',
    'SYY': 'Travel',
    # Energy
    'XOM': 'Energy', 'CVX': 'Energy', 'SLB': 'Energy', 'HAL': 'Energy', 'OXY': 'Energy',
    # Real Estate
    'SPG': 'RealEstate', 'VNO': 'RealEstate', 'BXP': 'RealEstate', 'MAC': 'RealEstate',
    # Industrials / Defence
    'RTX': 'Industrials', 'GE': 'Industrials',
    # Financials
    'JPM': 'Financials', 'BAC': 'Financials', 'WFC': 'Financials', 'C': 'Financials',
    'AXP': 'Financials', 'V': 'Financials', 'MA': 'Financials',
    # Consumer Discretionary
    'SBUX': 'ConsumerDisc', 'TSLA': 'ConsumerDisc',
}

print(f"Models: {len(MODELS)}")
print(f"Sectors: {len(set(SECTOR_MAP.values()))}")

## 0.1 Data Loading & Preprocessing

We load the FNSPID dataset (2019-2022), download price data, and compute T+1 forward returns.
The COVID-affected stock universe provides a natural experiment: stocks whose fortunes changed
dramatically due to the pandemic, making look-ahead bias particularly detectable.

In [ ]:
# COVID-affected stock universe (same as original experiment)
TOP_TICKERS = [
    'ZM', 'PTON', 'DOCU', 'NFLX', 'AMZN', 'SHOP', 'TDOC', 'NVDA', 'AMD', 'AAPL',
    'MSFT', 'GOOGL', 'META', 'ETSY', 'CHWY', 'W', 'ROKU', 'PYPL', 'SQ', 'TEAM',
    'CRWD', 'OKTA', 'ZS', 'TWLO', 'DDOG', 'FSLY', 'NET', 'SNOW', 'EBAY', 'CRM',
    'TGT', 'WMT', 'COST', 'HD', 'LOW', 'CLX', 'PG', 'KMB', 'DPZ', 'MRNA',
    'PFE', 'BNTX', 'JNJ', 'GILD', 'REGN', 'ABT', 'LLY', 'AZN', 'NVAX', 'CCL',
    'RCL', 'NCLH', 'DAL', 'UAL', 'AAL', 'LUV', 'BA', 'MAR', 'HLT', 'EXPE',
    'BKNG', 'ABNB', 'UBER', 'LYFT', 'DIS', 'AMC', 'CNK', 'LYV', 'MGM', 'CZR',
    'WYNN', 'LVS', 'DRI', 'PLAY', 'SYY', 'XOM', 'CVX', 'SLB', 'HAL', 'OXY',
    'SPG', 'VNO', 'BXP', 'MAC', 'GPS', 'M', 'JWN', 'KSS', 'URBN', 'RTX',
    'GE', 'JPM', 'BAC', 'WFC', 'C', 'AXP', 'V', 'MA', 'SBUX', 'TSLA'
]

# Load news
df_news = pd.read_csv("dataset/fnspid_2019_2022.csv", low_memory=False)
df_news['date'] = pd.to_datetime(df_news['Date']).dt.date
df_news = df_news[df_news['Stock_symbol'].isin(TOP_TICKERS)].copy()
df_news.sort_values('date', inplace=True)
print(f"Total news articles: {len(df_news)}")
print(f"Date range: {df_news['date'].min()} to {df_news['date'].max()}")
print(f"Unique tickers with news: {df_news['Stock_symbol'].nunique()}")

In [ ]:
# Download price data and compute T+1 forward returns
start_date = "2019-01-01"
end_date = "2023-01-01"

prices = yf.download(TOP_TICKERS, start=start_date, end=end_date)['Close']
returns = prices.pct_change(fill_method=None)
forward_returns = returns.shift(-1)

# Reshape to long format
fwd_ret_long = forward_returns.reset_index().melt(
    id_vars='Date', var_name='Stock_symbol', value_name='fwd_return'
)
fwd_ret_long['Date'] = fwd_ret_long['Date'].dt.date
fwd_ret_long.rename(columns={'Date': 'date'}, inplace=True)

# Merge
df_merged = pd.merge(df_news, fwd_ret_long, on=['date', 'Stock_symbol'], how='inner')
df_merged.dropna(subset=['fwd_return'], inplace=True)

# Add sector column
df_merged['sector'] = df_merged['Stock_symbol'].map(SECTOR_MAP).fillna('Other')

print(f"Merged dataset: {len(df_merged)} rows")
print(f"Date range: {df_merged['date'].min()} to {df_merged['date'].max()}")
print(f"\nSector distribution:")
print(df_merged['sector'].value_counts())

## 0.2 Embedding Extraction

Extract [CLS] embeddings for all models. This is the most time-consuming step.

In [ ]:
def extract_embeddings(texts, model_name, batch_size=32):
    """Extract [CLS] token embeddings from a BERT-style encoder."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding with {model_name}"):
            batch_texts = texts[i:i+batch_size].tolist()
            inputs = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=512, return_tensors="pt"
            ).to(device)
            outputs = model(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_emb)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

    return np.vstack(embeddings)

# Extract embeddings for all models
texts = df_merged['Article_title'].fillna("").values
for model_name, model_path in MODELS.items():
    print(f"\n--- Extracting: {model_name} ---")
    embs = extract_embeddings(texts, model_path)
    df_merged[model_name] = list(embs)
    print(f"  Shape: {embs.shape}")

print(f"\nAll embeddings extracted. DataFrame shape: {df_merged.shape}")

## 0.3 Shared Utility Functions

Core backtest logic and metric calculations used across all experiments.

In [ ]:
# ============================================================
# SHARED UTILITY FUNCTIONS
# ============================================================

def calculate_sharpe(returns_series):
    """Annualised Sharpe ratio for a daily return series."""
    mu = returns_series.mean()
    sigma = returns_series.std()
    return np.sqrt(252) * (mu / sigma) if sigma > 0 else 0.0

def calculate_max_drawdown(cum_returns):
    """Maximum drawdown from a cumulative return series."""
    wealth = 1 + cum_returns
    peak = wealth.cummax()
    drawdown = (wealth - peak) / peak
    return drawdown.min()

def calculate_sortino(returns_series):
    """Annualised Sortino ratio (downside deviation only)."""
    mu = returns_series.mean()
    downside = returns_series[returns_series < 0].std()
    return np.sqrt(252) * (mu / downside) if downside > 0 else 0.0

def calculate_calmar(returns_series):
    """Calmar ratio = annualised return / max drawdown."""
    cum = (1 + returns_series).cumprod() - 1
    ann_ret = returns_series.mean() * 252
    mdd = abs(calculate_max_drawdown(cum))
    return ann_ret / mdd if mdd > 0 else 0.0

def backtest_strategy(df, embedding_col, lookback_days=60, decile_pct=0.10,
                      ridge_alpha=1.0, model_cls=None, return_daily_detail=False):
    """
    Rolling long-short portfolio backtest.
    
    Parameters
    ----------
    df : DataFrame with embedding column, 'date', 'Stock_symbol', 'fwd_return'
    embedding_col : column name containing embedding vectors
    lookback_days : training window size
    decile_pct : fraction of stocks in long/short legs
    ridge_alpha : regularisation strength (used if model_cls is None or Ridge)
    model_cls : sklearn model class override (default: Ridge)
    return_daily_detail : if True, also returns per-day prediction DataFrame
    
    Returns
    -------
    portfolio_returns : pd.Series of daily long-short returns
    long_tickers : list of tickers in long leg each day
    short_tickers : list of tickers in short leg each day
    daily_details : (optional) list of per-day DataFrames with predictions
    """
    unique_dates = sorted(df['date'].unique())
    portfolio_returns = []
    long_tickers = []
    short_tickers = []
    daily_dates = []
    daily_details = []

    for i in range(lookback_days, len(unique_dates)):
        train_dates = unique_dates[i - lookback_days:i]
        test_date = unique_dates[i]

        train_data = df[df['date'].isin(train_dates)]
        test_data = df[df['date'] == test_date]

        if len(train_data) < 100 or len(test_data) < 20:
            continue

        X_train = np.vstack(train_data[embedding_col].values)
        y_train = train_data['fwd_return'].values
        X_test = np.vstack(test_data[embedding_col].values)

        # Fit model
        if model_cls is not None:
            mdl = model_cls
        else:
            mdl = Ridge(alpha=ridge_alpha)
        mdl.fit(X_train, y_train)
        predictions = mdl.predict(X_test)

        test_data = test_data.copy()
        test_data['predicted_return'] = predictions

        # Aggregate by ticker
        daily_preds = test_data.groupby('Stock_symbol').agg({
            'predicted_return': 'mean',
            'fwd_return': 'mean'
        })

        n_stocks = len(daily_preds)
        k = max(1, int(n_stocks * decile_pct))

        daily_preds = daily_preds.sort_values('predicted_return', ascending=False)
        long_port = daily_preds.head(k)
        short_port = daily_preds.tail(k)

        long_tickers.extend(long_port.index.tolist())
        short_tickers.extend(short_port.index.tolist())

        daily_return = long_port['fwd_return'].mean() - short_port['fwd_return'].mean()
        portfolio_returns.append(daily_return)
        daily_dates.append(test_date)

        if return_daily_detail:
            detail = daily_preds.copy()
            detail['date'] = test_date
            detail['long'] = detail.index.isin(long_port.index)
            detail['short'] = detail.index.isin(short_port.index)
            daily_details.append(detail)

    ret_series = pd.Series(portfolio_returns, index=pd.to_datetime(daily_dates))

    if return_daily_detail:
        return ret_series, long_tickers, short_tickers, daily_details
    return ret_series, long_tickers, short_tickers

def full_metrics(returns_series, name=""):
    """Compute a full suite of performance metrics."""
    sharpe = calculate_sharpe(returns_series)
    sortino = calculate_sortino(returns_series)
    calmar = calculate_calmar(returns_series)
    cum = (1 + returns_series).cumprod() - 1
    mdd = calculate_max_drawdown(cum)
    ann_ret = returns_series.mean() * 252
    ann_vol = returns_series.std() * np.sqrt(252)
    hit_rate = (returns_series > 0).mean()
    return {
        'Model': name,
        'Ann. Return': ann_ret,
        'Ann. Volatility': ann_vol,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Calmar': calmar,
        'Max Drawdown': mdd,
        'Hit Rate': hit_rate,
        'N Days': len(returns_series)
    }

# ============================================================
# SHARED CONSTANTS — defined here so all experiments can use them
# without depending on Experiment 2 having run first
# ============================================================
SUB_PERIODS = {
    'Pre-COVID':   (pd.to_datetime('2019-01-01'), pd.to_datetime('2020-02-18')),
    'COVID Crash': (pd.to_datetime('2020-02-19'), pd.to_datetime('2020-06-30')),
    'Recovery':    (pd.to_datetime('2020-07-01'), pd.to_datetime('2021-06-30')),
    'Post-COVID':  (pd.to_datetime('2021-07-01'), pd.to_datetime('2022-12-31')),
}

print("Utility functions and shared constants defined.")
print(f"SUB_PERIODS: {list(SUB_PERIODS.keys())}")

# ============================================================
# LAZY INITIALISER — ensures base_results is always available
# ============================================================
def ensure_base_results():
    """
    Populate base_results if it hasn't been built yet.
    Lets any experiment cell run stand-alone without requiring
    Experiment 1 to have executed first.
    """
    global base_results
    if 'base_results' not in globals() or not base_results:
        print("base_results not found — running base backtests now...")
        base_results = {}
        for model_name in MODELS.keys():
            print(f"  Backtesting: {model_name}")
            ret, longs, shorts = backtest_strategy(df_merged, model_name)
            base_results[model_name] = (ret, longs, shorts)
        print("base_results ready.")
    else:
        print("base_results already populated — skipping.")

base_results = {}  # initialise as empty; populated by Exp 1 or ensure_base_results()


---
## Experiment 1: Replication with Bootstrap Confidence Intervals

The original experiment reports a single Sharpe ratio per model, but a single backtest provides no
measure of statistical uncertainty. We address this with:

1. **Full performance table** (Sharpe, Sortino, Calmar, Max Drawdown, Hit Rate)
2. **Bootstrap confidence intervals** on the Sharpe ratio (10,000 resamples with replacement)
3. **Cumulative return plot** with the COVID crash period highlighted

This answers: *Are the observed Sharpe ratio differences statistically meaningful, or within sampling noise?*

In [ ]:
# ============================================================
# EXPERIMENT 1: Run base backtests for all models
# ============================================================
base_results = {}  # model_name -> (returns_series, longs, shorts)

for model_name in MODELS.keys():
    print(f"Backtesting: {model_name}")
    ret, longs, shorts = backtest_strategy(df_merged, model_name)
    base_results[model_name] = (ret, longs, shorts)

# Full performance table
metrics_rows = []
for model_name, (ret, _, _) in base_results.items():
    metrics_rows.append(full_metrics(ret, model_name))

metrics_df = pd.DataFrame(metrics_rows).set_index('Model')
print("\n" + "=" * 80)
print("FULL PERFORMANCE TABLE (Base Case: lookback=60, decile=10%, Ridge alpha=1.0)")
print("=" * 80)
metrics_df.style.format({
    'Ann. Return': '{:.4f}', 'Ann. Volatility': '{:.4f}',
    'Sharpe': '{:.3f}', 'Sortino': '{:.3f}', 'Calmar': '{:.3f}',
    'Max Drawdown': '{:.3f}', 'Hit Rate': '{:.3f}'
})
print(metrics_df.to_string(float_format=lambda x: f"{x:.4f}"))

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 1b: Bootstrap Confidence Intervals on Sharpe Ratio
# ============================================================
def bootstrap_sharpe(returns_series, n_bootstrap=10000, ci=0.95):
    """Bootstrap the Sharpe ratio and return point estimate + CI."""
    n = len(returns_series)
    boot_sharpes = np.zeros(n_bootstrap)
    vals = returns_series.values

    for b in range(n_bootstrap):
        sample = np.random.choice(vals, size=n, replace=True)
        mu = sample.mean()
        sigma = sample.std()
        boot_sharpes[b] = np.sqrt(252) * (mu / sigma) if sigma > 0 else 0.0

    alpha = (1 - ci) / 2
    lo = np.percentile(boot_sharpes, 100 * alpha)
    hi = np.percentile(boot_sharpes, 100 * (1 - alpha))
    return boot_sharpes, lo, hi

print("Bootstrap Sharpe Ratios (95% CI, 10,000 resamples)")
print("-" * 65)

boot_results = {}
for model_name, (ret, _, _) in base_results.items():
    boot_dist, lo, hi = bootstrap_sharpe(ret)
    point = calculate_sharpe(ret)
    boot_results[model_name] = (point, lo, hi, boot_dist)
    print(f"  {model_name:20s}  Sharpe = {point:+.3f}  [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 1c: Visualisations
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Panel A: Cumulative Returns ---
ax = axes[0]
for model_name, (ret, _, _) in base_results.items():
    cum = (1 + ret).cumprod() - 1
    ax.plot(cum.index, cum, label=model_name, color=MODEL_COLORS[model_name], linewidth=1.5)

# COVID crash shading
ax.axvspan(pd.to_datetime('2020-02-19'), pd.to_datetime('2020-03-23'),
           color='red', alpha=0.12, label='COVID-19 Crash')
ax.set_title('(a) Cumulative Returns of Long-Short Portfolio', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

# --- Panel B: Bootstrap Sharpe with CI ---
ax = axes[1]
model_names = list(boot_results.keys())
points = [boot_results[m][0] for m in model_names]
los = [boot_results[m][1] for m in model_names]
his = [boot_results[m][2] for m in model_names]
colors = [MODEL_COLORS[m] for m in model_names]

y_pos = np.arange(len(model_names))
xerr_lo = [p - l for p, l in zip(points, los)]
xerr_hi = [h - p for p, h in zip(points, his)]

ax.barh(y_pos, points, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.errorbar(points, y_pos, xerr=[xerr_lo, xerr_hi], fmt='none', ecolor='black',
            capsize=4, linewidth=1.5)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(model_names)
ax.set_xlabel('Annualised Sharpe Ratio')
ax.set_title('(b) Sharpe Ratio with 95% Bootstrap CI', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exp1_base_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: figures/exp1_base_results.png")

---
## Experiment 2: Temporal Sub-Period Decomposition

Look-ahead bias should manifest most strongly during regime changes. We split the 2019-2022
test period into four economically meaningful sub-periods:

| Period | Dates | Rationale |
|--------|-------|-----------|
| **Pre-COVID** | 2019-01 to 2020-02-18 | Normal market, no pandemic signal |
| **COVID Crash** | 2020-02-19 to 2020-06-30 | Acute crisis + initial recovery |
| **Recovery / Vaccine** | 2020-07-01 to 2021-06-30 | Vaccine rollout, stimulus-driven rally |
| **Post-COVID** | 2021-07-01 to 2022-12 | Normalisation, rate hikes, tech drawdown |

**Hypothesis:** Chronologically inconsistent models (2021/2022) should show *anomalously* different
performance during the COVID Crash and Recovery periods — precisely when their training data
contains information about pandemic outcomes that pre-2019 models lack.

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 2: Temporal Sub-Period Decomposition
# ============================================================
SUB_PERIODS = {
    'Pre-COVID':       (pd.to_datetime('2019-01-01'), pd.to_datetime('2020-02-18')),
    'COVID Crash':     (pd.to_datetime('2020-02-19'), pd.to_datetime('2020-06-30')),
    'Recovery':        (pd.to_datetime('2020-07-01'), pd.to_datetime('2021-06-30')),
    'Post-COVID':      (pd.to_datetime('2021-07-01'), pd.to_datetime('2022-12-31')),
}

period_metrics = []

for model_name, (ret, _, _) in base_results.items():
    for period_name, (start, end) in SUB_PERIODS.items():
        mask = (ret.index >= start) & (ret.index <= end)
        period_ret = ret[mask]
        if len(period_ret) < 10:
            continue
        sharpe = calculate_sharpe(period_ret)
        ann_ret = period_ret.mean() * 252
        period_metrics.append({
            'Model': model_name,
            'Period': period_name,
            'Sharpe': sharpe,
            'Ann. Return': ann_ret,
            'N Days': len(period_ret),
            'Hit Rate': (period_ret > 0).mean()
        })

period_df = pd.DataFrame(period_metrics)
pivot_sharpe = period_df.pivot(index='Model', columns='Period', values='Sharpe')
pivot_sharpe = pivot_sharpe[['Pre-COVID', 'COVID Crash', 'Recovery', 'Post-COVID']]

print("Sharpe Ratios by Sub-Period")
print("=" * 70)
print(pivot_sharpe.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ============================================================
# EXPERIMENT 2b: Grouped bar chart — Sharpe by period
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

periods = ['Pre-COVID', 'COVID Crash', 'Recovery', 'Post-COVID']
x = np.arange(len(periods))
width = 0.12
n_models = len(MODELS)

for i, model_name in enumerate(MODELS.keys()):
    vals = [pivot_sharpe.loc[model_name, p] if model_name in pivot_sharpe.index else 0
            for p in periods]
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=model_name,
           color=MODEL_COLORS[model_name], edgecolor='black', linewidth=0.5)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(periods, fontsize=11)
ax.set_ylabel('Annualised Sharpe Ratio', fontsize=11)
ax.set_title('Experiment 2: Sharpe Ratio by Market Regime', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exp2_period_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 2c: Rolling 63-day Sharpe (regime dynamics)
# ============================================================
fig, ax = plt.subplots(figsize=(15, 6))
window = 63  # ~3 months

for model_name, (ret, _, _) in base_results.items():
    rolling_mean = ret.rolling(window).mean()
    rolling_std = ret.rolling(window).std()
    rolling_sharpe = np.sqrt(252) * rolling_mean / rolling_std
    ax.plot(rolling_sharpe.index, rolling_sharpe, label=model_name,
            color=MODEL_COLORS[model_name], linewidth=1.2, alpha=0.85)

# Regime shading
for period_name, (start, end) in SUB_PERIODS.items():
    if period_name == 'COVID Crash':
        ax.axvspan(start, end, color='red', alpha=0.08)
    elif period_name == 'Recovery':
        ax.axvspan(start, end, color='green', alpha=0.05)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Experiment 2: Rolling 63-Day Sharpe Ratio', fontsize=13)
ax.set_ylabel('Annualised Sharpe (63d rolling)')
ax.set_xlabel('Date')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('figures/exp2_rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Experiment 3: Robustness to Strategy Parameters

A finding that only holds under one specific parameter configuration is fragile. We systematically vary:

1. **Lookback window**: 30, 60, 120, 252 trading days
2. **Decile size**: 5%, 10%, 20% of stocks in long/short legs
3. **Ridge alpha**: 0.01, 0.1, 1.0, 10.0, 100.0
4. **Prediction model**: Ridge, Lasso, Random Forest, Gradient Boosting

For each variation, we compare the Sharpe ratio of chronologically consistent (2017/2018 avg)
vs inconsistent (2021/2022 avg) models. If look-ahead bias is real, the gap should persist
across parameter choices.

In [ ]:
# ============================================================
# EXPERIMENT 3a: Lookback Window Sensitivity
# ============================================================
LOOKBACKS = [30, 60, 120, 252]
# Only test representative models to save compute
TEST_MODELS = ['ChronoBERT_2018', 'ChronoBERT_2022', 'Standard_BERT', 'FinBERT']

lookback_results = []

for lb in LOOKBACKS:
    for model_name in TEST_MODELS:
        print(f"  lookback={lb}, model={model_name}")
        ret, _, _ = backtest_strategy(df_merged, model_name, lookback_days=lb)
        sharpe = calculate_sharpe(ret)
        lookback_results.append({
            'Lookback': lb, 'Model': model_name, 'Sharpe': sharpe,
            'Ann. Return': ret.mean() * 252, 'N Days': len(ret)
        })

lb_df = pd.DataFrame(lookback_results)
pivot_lb = lb_df.pivot(index='Model', columns='Lookback', values='Sharpe')
print("\nSharpe by Lookback Window")
print(pivot_lb.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ============================================================
# EXPERIMENT 3b: Decile Size Sensitivity
# ============================================================
DECILES = [0.05, 0.10, 0.20]

decile_results = []
for dec in DECILES:
    for model_name in TEST_MODELS:
        print(f"  decile={dec:.0%}, model={model_name}")
        ret, _, _ = backtest_strategy(df_merged, model_name, decile_pct=dec)
        sharpe = calculate_sharpe(ret)
        decile_results.append({
            'Decile': f"{dec:.0%}", 'Model': model_name, 'Sharpe': sharpe,
            'Ann. Return': ret.mean() * 252
        })

dec_df = pd.DataFrame(decile_results)
pivot_dec = dec_df.pivot(index='Model', columns='Decile', values='Sharpe')
print("\nSharpe by Decile Size")
print(pivot_dec.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ============================================================
# EXPERIMENT 3c: Ridge Alpha Sensitivity
# ============================================================
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]

alpha_results = []
for alpha in ALPHAS:
    for model_name in TEST_MODELS:
        print(f"  alpha={alpha}, model={model_name}")
        ret, _, _ = backtest_strategy(df_merged, model_name, ridge_alpha=alpha)
        sharpe = calculate_sharpe(ret)
        alpha_results.append({
            'Alpha': alpha, 'Model': model_name, 'Sharpe': sharpe
        })

alpha_df = pd.DataFrame(alpha_results)
pivot_alpha = alpha_df.pivot(index='Model', columns='Alpha', values='Sharpe')
print("\nSharpe by Ridge Alpha")
print(pivot_alpha.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ============================================================
# EXPERIMENT 3d: Prediction Model Sensitivity
# ============================================================
PRED_MODELS = {
    'Ridge':            Ridge(alpha=1.0),
    'Lasso':            Lasso(alpha=0.001, max_iter=5000),
    'RandomForest':     RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42),
}

model_type_results = []
for pred_name, pred_cls in PRED_MODELS.items():
    for emb_name in TEST_MODELS:
        print(f"  predictor={pred_name}, embeddings={emb_name}")
        ret, _, _ = backtest_strategy(df_merged, emb_name, model_cls=pred_cls)
        sharpe = calculate_sharpe(ret)
        model_type_results.append({
            'Predictor': pred_name, 'Embedding': emb_name, 'Sharpe': sharpe
        })

mt_df = pd.DataFrame(model_type_results)
pivot_mt = mt_df.pivot(index='Embedding', columns='Predictor', values='Sharpe')
print("\nSharpe by Prediction Model")
print(pivot_mt.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ============================================================
# EXPERIMENT 3e: Summary heatmaps for all robustness checks
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# (a) Lookback
ax = axes[0, 0]
sns.heatmap(pivot_lb, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Sharpe'})
ax.set_title('(a) Lookback Window (days)', fontsize=12)

# (b) Decile
ax = axes[0, 1]
sns.heatmap(pivot_dec, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Sharpe'})
ax.set_title('(b) Decile Size (% of universe)', fontsize=12)

# (c) Ridge Alpha
ax = axes[1, 0]
sns.heatmap(pivot_alpha, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Sharpe'})
ax.set_title('(c) Ridge Regularisation (alpha)', fontsize=12)

# (d) Prediction Model
ax = axes[1, 1]
sns.heatmap(pivot_mt, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Sharpe'})
ax.set_title('(d) Prediction Model', fontsize=12)

plt.suptitle('Experiment 3: Robustness of Sharpe Ratio Across Strategy Parameters',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/exp3_robustness_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Experiment 4: Embedding Geometry & Divergence Analysis

If look-ahead bias changes model behaviour, it should be visible in the embedding space itself —
not just in downstream portfolio performance. We measure:

1. **Pairwise cosine similarity** between the same article embedded by different model vintages
2. **Embedding divergence by sector** — which sectors see the largest shifts?
3. **PCA projection** — visualise how the same headlines occupy different regions of embedding space

This provides a model-intrinsic (not strategy-dependent) measure of temporal inconsistency.

In [ ]:
# ============================================================
# EXPERIMENT 4a: Pairwise Cosine Similarity Between Vintages
# ============================================================
# For the SAME article, how similar are embeddings from different model vintages?
# High similarity = models agree; low similarity = temporal drift in representation.

from sklearn.metrics.pairwise import cosine_similarity

# Sample articles for tractable computation (use all if dataset is small enough)
np.random.seed(42)
sample_idx = np.random.choice(len(df_merged), size=min(5000, len(df_merged)), replace=False)

model_names = list(MODELS.keys())
n_models = len(model_names)

# Build embedding matrices for sampled articles
emb_matrices = {}
for mname in model_names:
    emb_matrices[mname] = np.vstack(df_merged.iloc[sample_idx][mname].values)

# Compute pairwise mean cosine similarity
sim_matrix = np.zeros((n_models, n_models))
for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        # Row-wise cosine similarity between paired embeddings
        sims = np.array([
            cosine_similarity(emb_matrices[m1][k:k+1], emb_matrices[m2][k:k+1])[0, 0]
            for k in range(len(sample_idx))
        ])
        sim_matrix[i, j] = sims.mean()

sim_df = pd.DataFrame(sim_matrix, index=model_names, columns=model_names)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_df, annot=True, fmt='.4f', cmap='YlOrRd', vmin=0.5, vmax=1.0,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Mean Cosine Similarity'})
ax.set_title('Experiment 4a: Pairwise Cosine Similarity of Same-Article Embeddings', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exp4a_cosine_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observations:")
print(f"  ChronoBERT_2017 vs 2018 similarity: {sim_df.loc['ChronoBERT_2017','ChronoBERT_2018']:.4f}")
print(f"  ChronoBERT_2017 vs 2022 similarity: {sim_df.loc['ChronoBERT_2017','ChronoBERT_2022']:.4f}")
print(f"  ChronoBERT_2018 vs 2022 similarity: {sim_df.loc['ChronoBERT_2018','ChronoBERT_2022']:.4f}")
print(f"  Standard_BERT vs FinBERT similarity: {sim_df.loc['Standard_BERT','FinBERT']:.4f}")

In [ ]:
# ============================================================
# EXPERIMENT 4b: Embedding Divergence Over Time
# ============================================================
# Track how cosine similarity between ChronoBERT_2018 and ChronoBERT_2022
# embeddings evolves over the test period. If look-ahead bias matters,
# divergence should increase around COVID events.

from sklearn.metrics.pairwise import paired_cosine_distances

# Compute per-article cosine distance between 2018 and 2022 vintages
df_merged_sorted = df_merged.sort_values('date').copy()
emb_2018 = np.vstack(df_merged_sorted['ChronoBERT_2018'].values)
emb_2022 = np.vstack(df_merged_sorted['ChronoBERT_2022'].values)

# Cosine similarity = 1 - cosine_distance
cos_dist = paired_cosine_distances(emb_2018, emb_2022)
df_merged_sorted['cos_dist_2018_vs_2022'] = cos_dist

# Daily average divergence
daily_div = df_merged_sorted.groupby('date')['cos_dist_2018_vs_2022'].mean()
daily_div.index = pd.to_datetime(daily_div.index)

# Rolling 21-day average for smoothing
rolling_div = daily_div.rolling(21, min_periods=5).mean()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(rolling_div.index, rolling_div, color='darkred', linewidth=1.5,
        label='21-day rolling mean cosine distance')
ax.fill_between(rolling_div.index, rolling_div, alpha=0.2, color='red')

# Regime shading
for period_name, (start, end) in SUB_PERIODS.items():
    if period_name == 'COVID Crash':
        ax.axvspan(start, end, color='red', alpha=0.08, label='COVID Crash')
    elif period_name == 'Recovery':
        ax.axvspan(start, end, color='green', alpha=0.05, label='Recovery')

ax.set_title('Experiment 4b: Embedding Divergence (ChronoBERT 2018 vs 2022) Over Time', fontsize=12)
ax.set_ylabel('Mean Cosine Distance')
ax.set_xlabel('Date')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('figures/exp4b_temporal_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 4c: PCA Projection of Embeddings
# ============================================================
# Visualise how the same headlines land in different regions of
# embedding space across model vintages.

from sklearn.decomposition import PCA

# Use a smaller sample for visualisation clarity
vis_idx = np.random.choice(len(df_merged), size=min(2000, len(df_merged)), replace=False)

# Stack embeddings from two vintages for joint PCA
emb_early = np.vstack(df_merged.iloc[vis_idx]['ChronoBERT_2018'].values)
emb_late = np.vstack(df_merged.iloc[vis_idx]['ChronoBERT_2022'].values)
combined = np.vstack([emb_early, emb_late])

pca = PCA(n_components=2, random_state=42)
projected = pca.fit_transform(combined)
n = len(vis_idx)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(projected[:n, 0], projected[:n, 1], alpha=0.3, s=8,
           color=MODEL_COLORS['ChronoBERT_2018'], label='ChronoBERT_2018')
ax.scatter(projected[n:, 0], projected[n:, 1], alpha=0.3, s=8,
           color=MODEL_COLORS['ChronoBERT_2022'], label='ChronoBERT_2022')

ax.set_title('Experiment 4c: PCA of Same-Article Embeddings (2018 vs 2022)', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(markerscale=5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exp4c_pca_projection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}")

---
## Experiment 5: Year-Matched Chronological Consistency Evaluation

The core claim of the ChronoBERT paper is that you should use the model checkpoint whose
training cutoff *predates* the data being analysed. This experiment operationalises that:

- **Correct protocol ("Year-Matched"):** For 2019 news use ChronoBERT_2018, for 2020 news
  use ChronoBERT_2018 (still the most recent pre-data checkpoint), for 2021 news use
  ChronoBERT_2018, for 2022 news use ChronoBERT_2021.
- **Incorrect protocol ("Always-Latest"):** Always use ChronoBERT_2022 regardless of year.
- **Incorrect protocol ("Always-Oldest"):** Always use ChronoBERT_2017 regardless of year.

We compare strategy performance of the year-matched approach vs the two naïve approaches.

In [ ]:
# ============================================================
# EXPERIMENT 5: Year-Matched Chronological Consistency
# ============================================================

# Define the year-to-model mapping (use most recent PRE-data checkpoint)
YEAR_MODEL_MAP = {
    2019: 'ChronoBERT_2018',
    2020: 'ChronoBERT_2018',
    2021: 'ChronoBERT_2018',
    2022: 'ChronoBERT_2021',
}

# Create a "year-matched" embedding column: for each row, pick the
# embedding from the chronologically appropriate model
def get_year_matched_embedding(row):
    year = row['date'].year if hasattr(row['date'], 'year') else pd.to_datetime(row['date']).year
    model_col = YEAR_MODEL_MAP.get(year, 'ChronoBERT_2018')
    return row[model_col]

print("Building year-matched embedding column...")
df_merged['YearMatched'] = df_merged.apply(get_year_matched_embedding, axis=1)

# Run backtests for all three protocols
protocols = {
    'Year-Matched': 'YearMatched',
    'Always-Latest (2022)': 'ChronoBERT_2022',
    'Always-Oldest (2017)': 'ChronoBERT_2017',
    'Always-2018': 'ChronoBERT_2018',
}

protocol_results = {}
for proto_name, col_name in protocols.items():
    print(f"  Backtesting: {proto_name}")
    ret, longs, shorts = backtest_strategy(df_merged, col_name)
    protocol_results[proto_name] = ret

# Metrics table
proto_metrics = []
for proto_name, ret in protocol_results.items():
    proto_metrics.append(full_metrics(ret, proto_name))

proto_metrics_df = pd.DataFrame(proto_metrics).set_index('Model')
print("\n=== Year-Matched vs Naïve Protocols ===")
print(proto_metrics_df.to_string(float_format=lambda x: f"{x:.4f}"))

In [ ]:
# ============================================================
# EXPERIMENT 5b: Cumulative Returns — Year-Matched vs Naïve Protocols
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

proto_colors = {
    'Year-Matched': '#2ca02c',
    'Always-Latest (2022)': '#d62728',
    'Always-Oldest (2017)': '#1f77b4',
    'Always-2018': '#ff7f0e',
}

for proto_name, ret in protocol_results.items():
    cum = (1 + ret).cumprod() - 1
    style = '-' if proto_name == 'Year-Matched' else '--'
    lw = 2.5 if proto_name == 'Year-Matched' else 1.5
    ax.plot(cum.index, cum, label=proto_name, linestyle=style, linewidth=lw,
            color=proto_colors.get(proto_name, 'gray'))

# Regime shading
for period_name, (start, end) in SUB_PERIODS.items():
    if period_name == 'COVID Crash':
        ax.axvspan(start, end, color='red', alpha=0.06)

ax.set_title('Experiment 5: Year-Matched vs Naïve Embedding Protocols', fontsize=13)
ax.set_ylabel('Cumulative Return')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('figures/exp5_year_matched.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Experiment 6: Sector-Level Decomposition of Look-Ahead Bias

The COVID pandemic affected sectors asymmetrically. Remote-work tech stocks surged while
travel and energy collapsed. If look-ahead bias is real, we expect:

- **Tech/Remote-Work:** Later vintages should systematically push these tickers into long legs
- **Travel/Hospitality:** Later vintages should push these into short legs (avoiding them)
- **Healthcare/Pharma:** Vaccine-related tickers (BNTX, MRNA, PFE) should show dramatic shifts

We define sector groupings and measure per-sector long/short frequency differences between
the 2018 (consistent) and 2022 (inconsistent) models.

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 6: Sector-Level Decomposition
# ============================================================

SECTOR_MAP = {
    # Tech / Remote Work (COVID winners)
    'Tech_Remote': ['ZM', 'PTON', 'DOCU', 'NFLX', 'AMZN', 'SHOP', 'TDOC', 'NVDA', 'AMD',
                    'AAPL', 'MSFT', 'GOOGL', 'META', 'ETSY', 'CHWY', 'W', 'ROKU', 'PYPL',
                    'SQ', 'TEAM', 'CRWD', 'OKTA', 'ZS', 'TWLO', 'DDOG', 'FSLY', 'NET',
                    'SNOW', 'EBAY', 'CRM', 'TSLA'],
    # Travel / Hospitality (COVID losers)
    'Travel_Hosp': ['CCL', 'RCL', 'NCLH', 'DAL', 'UAL', 'AAL', 'LUV', 'BA', 'MAR', 'HLT',
                    'EXPE', 'BKNG', 'ABNB', 'UBER', 'LYFT', 'DIS', 'AMC', 'CNK', 'LYV',
                    'MGM', 'CZR', 'WYNN', 'LVS', 'DRI', 'PLAY'],
    # Healthcare / Pharma (Vaccine plays)
    'Healthcare':  ['MRNA', 'PFE', 'BNTX', 'JNJ', 'GILD', 'REGN', 'ABT', 'LLY', 'AZN', 'NVAX'],
    # Consumer / Retail
    'Consumer':    ['TGT', 'WMT', 'COST', 'HD', 'LOW', 'CLX', 'PG', 'KMB', 'DPZ',
                    'SYY', 'GPS', 'M', 'JWN', 'KSS', 'URBN', 'SBUX'],
    # Energy (oil crash)
    'Energy':      ['XOM', 'CVX', 'SLB', 'HAL', 'OXY'],
    # Financials / Real Estate
    'Fin_RE':      ['JPM', 'BAC', 'WFC', 'C', 'AXP', 'V', 'MA', 'SPG', 'VNO', 'BXP', 'MAC'],
}

# Invert to ticker -> sector
ticker_to_sector = {}
for sector, tickers in SECTOR_MAP.items():
    for t in tickers:
        ticker_to_sector[t] = sector

# Count long/short appearances per sector for 2018 vs 2022
def sector_frequency(ticker_list):
    """Count sector appearances from a list of tickers."""
    sectors = [ticker_to_sector.get(t, 'Other') for t in ticker_list]
    return pd.Series(sectors).value_counts()

compare_models = ['ChronoBERT_2018', 'ChronoBERT_2022']
sector_freq_data = {}

for mname in compare_models:
    _, longs, shorts = base_results[mname]
    long_sectors = sector_frequency(longs)
    short_sectors = sector_frequency(shorts)
    sector_freq_data[f'{mname}_long'] = long_sectors
    sector_freq_data[f'{mname}_short'] = short_sectors

sector_freq_df = pd.DataFrame(sector_freq_data).fillna(0).astype(int)

# Normalise to percentages
for col in sector_freq_df.columns:
    sector_freq_df[col + '_pct'] = sector_freq_df[col] / sector_freq_df[col].sum() * 100

print("Sector Frequency in Long/Short Legs (raw counts)")
print("=" * 70)
print(sector_freq_df[[c for c in sector_freq_df.columns if '_pct' not in c]].to_string())
print("\nSector Frequency (% of total)")
print(sector_freq_df[[c for c in sector_freq_df.columns if '_pct' in c]].to_string(float_format=lambda x: f"{x:.1f}%"))

In [ ]:
# ============================================================
# EXPERIMENT 6b: Sector Long-Short Bias Shift (2018 vs 2022)
# ============================================================
# Positive = model puts sector into long more; Negative = model avoids it (pushes to short)
# We compute: (long_pct_2022 - long_pct_2018) - (short_pct_2022 - short_pct_2018)

sectors = list(SECTOR_MAP.keys())
bias_shift = []

for sector in sectors:
    l18 = sector_freq_df.loc[sector, 'ChronoBERT_2018_long_pct'] if sector in sector_freq_df.index else 0
    l22 = sector_freq_df.loc[sector, 'ChronoBERT_2022_long_pct'] if sector in sector_freq_df.index else 0
    s18 = sector_freq_df.loc[sector, 'ChronoBERT_2018_short_pct'] if sector in sector_freq_df.index else 0
    s22 = sector_freq_df.loc[sector, 'ChronoBERT_2022_short_pct'] if sector in sector_freq_df.index else 0
    
    long_shift = l22 - l18
    short_shift = s22 - s18
    net_bias = long_shift - short_shift
    
    bias_shift.append({
        'Sector': sector,
        'Long Shift (pp)': long_shift,
        'Short Shift (pp)': short_shift,
        'Net Bias Shift': net_bias
    })

bias_df = pd.DataFrame(bias_shift).set_index('Sector').sort_values('Net Bias Shift', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in bias_df['Net Bias Shift']]
bias_df['Net Bias Shift'].plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Net Bias Shift (percentage points)\n(+) = 2022 model favours sector more, (-) = 2022 model avoids sector')
ax.set_title('Experiment 6: Sector-Level Bias Shift (ChronoBERT 2022 vs 2018)', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exp6_sector_bias.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBias shift table:")
print(bias_df.to_string(float_format=lambda x: f"{x:+.2f}"))

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 6c: Per-Ticker Top Movers — Who shifted most?
# ============================================================
# Identify individual tickers whose long/short frequency changed
# the most between ChronoBERT_2018 and ChronoBERT_2022

_, longs_2018, shorts_2018 = base_results['ChronoBERT_2018']
_, longs_2022, shorts_2022 = base_results['ChronoBERT_2022']

long_counts_2018 = pd.Series(longs_2018).value_counts()
long_counts_2022 = pd.Series(longs_2022).value_counts()
short_counts_2018 = pd.Series(shorts_2018).value_counts()
short_counts_2022 = pd.Series(shorts_2022).value_counts()

all_tickers = sorted(set(long_counts_2018.index) | set(long_counts_2022.index) |
                      set(short_counts_2018.index) | set(short_counts_2022.index))

ticker_shifts = []
for t in all_tickers:
    l18 = long_counts_2018.get(t, 0)
    l22 = long_counts_2022.get(t, 0)
    s18 = short_counts_2018.get(t, 0)
    s22 = short_counts_2022.get(t, 0)
    net_shift = (l22 - l18) - (s22 - s18)
    ticker_shifts.append({
        'Ticker': t,
        'Sector': ticker_to_sector.get(t, 'Other'),
        'Long_2018': l18, 'Long_2022': l22,
        'Short_2018': s18, 'Short_2022': s22,
        'Net_Shift': net_shift
    })

ticker_shift_df = pd.DataFrame(ticker_shifts).set_index('Ticker').sort_values('Net_Shift', ascending=False)

print("Top 10 tickers the 2022 model FAVOURS more than 2018:")
print(ticker_shift_df.head(10).to_string())
print("\nTop 10 tickers the 2022 model AVOIDS more than 2018:")
print(ticker_shift_df.tail(10).to_string())

---
## Experiment 7: Permutation Tests for Statistical Significance

The Sharpe ratio differences observed so far could be due to chance. We use a permutation test
to assess statistical significance:

**Null hypothesis (H₀):** The daily returns from the ChronoBERT_2018 strategy and the
ChronoBERT_2022 strategy are drawn from the same distribution — any observed difference in
Sharpe ratios is due to random variation.

**Procedure:** On each permutation, for each day we randomly swap (or not) the returns between
the two strategies, recompute the Sharpe difference, and build an empirical null distribution.
The p-value is the fraction of permutations that produce a Sharpe difference at least as extreme
as the observed one.

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 7a: Permutation Test — Sharpe Difference
# ============================================================

def permutation_test_sharpe(ret_a, ret_b, n_permutations=10000, seed=42):
    """
    Permutation test for the difference in Sharpe ratios between two strategies.
    For each permutation, randomly swap daily returns between strategies.
    """
    rng = np.random.RandomState(seed)
    
    # Align on common dates
    common_idx = ret_a.index.intersection(ret_b.index)
    a = ret_a.loc[common_idx].values
    b = ret_b.loc[common_idx].values
    n = len(common_idx)
    
    observed_diff = calculate_sharpe(pd.Series(a)) - calculate_sharpe(pd.Series(b))
    
    perm_diffs = np.zeros(n_permutations)
    for p in tqdm(range(n_permutations), desc="Permutation test"):
        swap = rng.rand(n) > 0.5
        perm_a = np.where(swap, b, a)
        perm_b = np.where(swap, a, b)
        perm_diffs[p] = calculate_sharpe(pd.Series(perm_a)) - calculate_sharpe(pd.Series(perm_b))
    
    # Two-sided p-value
    p_value = (np.abs(perm_diffs) >= np.abs(observed_diff)).mean()
    
    return observed_diff, perm_diffs, p_value

# Test all ChronoBERT pairs against the 2018 baseline
ret_baseline = base_results['ChronoBERT_2018'][0]

perm_results = {}
for mname in ['ChronoBERT_2017', 'ChronoBERT_2021', 'ChronoBERT_2022', 'Standard_BERT', 'FinBERT']:
    ret_other = base_results[mname][0]
    obs_diff, perm_diffs, p_val = permutation_test_sharpe(ret_baseline, ret_other)
    perm_results[mname] = (obs_diff, perm_diffs, p_val)
    print(f"\nChronoBERT_2018 vs {mname}:")
    print(f"  Observed Sharpe difference: {obs_diff:+.4f}")
    print(f"  p-value (two-sided): {p_val:.4f}")
    print(f"  {'*** Significant at 5%' if p_val < 0.05 else '    Not significant at 5%'}")

In [ ]:
# ============================================================
# EXPERIMENT 7b: Null Distribution Histograms
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

key_comparisons = ['ChronoBERT_2022', 'Standard_BERT', 'FinBERT']

for idx, mname in enumerate(key_comparisons):
    ax = axes[idx]
    obs_diff, perm_diffs, p_val = perm_results[mname]
    
    ax.hist(perm_diffs, bins=80, density=True, alpha=0.7, color='steelblue',
            edgecolor='white', linewidth=0.3)
    ax.axvline(x=obs_diff, color='red', linewidth=2, linestyle='--',
               label=f'Observed: {obs_diff:+.3f}')
    ax.axvline(x=-obs_diff, color='red', linewidth=1, linestyle=':', alpha=0.5)
    
    ax.set_title(f'2018 vs {mname}\np = {p_val:.4f}', fontsize=11)
    ax.set_xlabel('Permuted Sharpe Difference')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Experiment 7: Permutation Test Null Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/exp7_permutation_tests.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ensure_base_results()

# ============================================================
# EXPERIMENT 7c: Paired t-test on Daily Return Differences
# ============================================================
# Complementary parametric test (under normality assumption)
from scipy import stats

print("Paired t-tests: ChronoBERT_2018 vs each model (daily return differences)")
print("=" * 70)

ret_2018 = base_results['ChronoBERT_2018'][0]

for mname in ['ChronoBERT_2017', 'ChronoBERT_2021', 'ChronoBERT_2022', 'Standard_BERT', 'FinBERT']:
    ret_other = base_results[mname][0]
    common = ret_2018.index.intersection(ret_other.index)
    diff = ret_2018.loc[common].values - ret_other.loc[common].values
    t_stat, p_val = stats.ttest_1samp(diff, 0)
    mean_diff = diff.mean() * 252  # annualise
    print(f"\n  vs {mname}:")
    print(f"    Annualised mean difference: {mean_diff:+.4f}")
    print(f"    t-statistic: {t_stat:.3f}, p-value: {p_val:.4f}")
    print(f"    {'*** Significant at 5%' if p_val < 0.05 else '    Not significant at 5%'}")

---
# Reasoning & Discussion

## Why These Seven Experiments?

The original notebook ran a single backtest with one parameter configuration and one stock universe.
While directionally interesting, a single Sharpe ratio comparison is insufficient for a research paper.
Below is the reasoning for each experiment:

### Experiment 1: Bootstrap Confidence Intervals
**Problem solved:** A point-estimate Sharpe ratio (e.g., 0.42 vs 0.38) tells you nothing about
whether the difference is meaningful or just noise. Two strategies can have nearly identical
expected performance yet produce different sample Sharpes due to estimation uncertainty.

**Why it matters:** Bootstrap CIs give us error bars. If the 95% CIs overlap heavily, the
performance "gap" between chronologically consistent and inconsistent models is not reliably
different from zero — which is itself an important finding.

---

### Experiment 2: Temporal Sub-Period Decomposition
**Problem solved:** Aggregate metrics can mask regime-dependent effects. A model could outperform
on average but catastrophically underperform during crises — or vice versa.

**Why it matters:** The core claim of ChronoBERT is that look-ahead bias matters because models
encode future knowledge. That knowledge is most valuable during *regime changes* (COVID crash,
vaccine rally). By splitting into sub-periods, we test whether the bias manifests where the
paper's theory predicts it should — not just "on average."

---

### Experiment 3: Robustness to Strategy Parameters
**Problem solved:** A finding that only holds under lookback=60, decile=10%, alpha=1.0, and
Ridge regression is a fragile artefact of one particular backtesting setup.

**Why it matters:** Reviewers will (rightly) ask: "Does this result depend on your specific
parameter choices?" By sweeping lookback windows, portfolio concentration, regularisation
strength, and even the prediction model class, we show whether the chronological consistency
effect is robust or parameter-dependent. This is standard practice in quantitative finance
research (Harvey et al., 2016).

---

### Experiment 4: Embedding Geometry & Divergence
**Problem solved:** Experiments 1-3 measure *downstream* effects (portfolio returns). But the
claim is about *embeddings* being contaminated. We need to verify that the embeddings actually
differ in a measurable way.

**Why it matters:** This is a model-intrinsic test, independent of the trading strategy. If
the cosine similarity between 2018 and 2022 embeddings of the same article is 0.999, then any
downstream performance difference is unlikely to be caused by look-ahead bias in the embeddings.
Conversely, if similarity is low and varies over time (especially around COVID), it directly
supports the paper's mechanism.

---

### Experiment 5: Year-Matched Chronological Consistency
**Problem solved:** The original notebook used the same model for all years. But the paper
recommends using the most recent checkpoint that *predates* the data. We need to test whether
following this protocol actually matters.

**Why it matters:** This is the *actionable recommendation* of the paper. If a researcher were
to adopt ChronoBERT, they would use the year-matched protocol. Comparing it against
"always-latest" and "always-oldest" directly measures the practical value of chronological
consistency.

---

### Experiment 6: Sector-Level Decomposition
**Problem solved:** Aggregate ticker frequency tables (as in the original notebook) are hard
to interpret. Grouping by economically meaningful sectors reveals *why* the model behaves
differently.

**Why it matters:** If the 2022 model systematically moves vaccine stocks (BNTX, MRNA) into
long positions and travel stocks into short positions — compared to the 2018 model — that is
direct, interpretable evidence of look-ahead bias. Sector-level analysis transforms
"the models differ" into "the models differ *because* the later one knows about COVID outcomes."

---

### Experiment 7: Permutation Tests
**Problem solved:** All prior experiments produce point estimates and visual comparisons. We
need a formal statistical test with a p-value.

**Why it matters:** The permutation test is non-parametric — it makes no distributional
assumptions about daily returns (which are known to be fat-tailed). The paired t-test provides
a parametric complement. Together, they let us state whether the observed differences are
statistically significant at conventional levels (p < 0.05).

---

## Summary of Evidence Structure

| Layer | What it tests | Experiment |
|-------|--------------|------------|
| **Statistical validity** | Are differences real or noise? | 1 (Bootstrap), 7 (Permutation) |
| **Temporal structure** | When does bias manifest? | 2 (Sub-periods) |
| **Robustness** | Is it parameter-dependent? | 3 (Sensitivity) |
| **Mechanistic** | Do embeddings actually differ? | 4 (Geometry) |
| **Practical** | Does the recommended protocol help? | 5 (Year-matched) |
| **Interpretive** | Which sectors/tickers drive it? | 6 (Sector decomposition) |

This multi-layered approach ensures that findings survive scrutiny from different angles —
statistical, economic, and mechanistic — which is what distinguishes rigorous empirical
research from a single-experiment demonstration.